# MLOps Pipeline — Validation

Validates the MLOps training pipeline output by checking:
- Latest training job status and metrics
- Model exists in SageMaker
- Endpoint is InService and responds to inference

In [ ]:
# Parameters
training_job_prefix = "bank-mktg-af-xgb"
model_prefix = "bank-mktg-af-model"
endpoint_name = "bank-mktg-af-prediction-endpoint"
region = "us-east-1"

In [ ]:
import boto3
import json
import os

region = region or os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
sm = boto3.client('sagemaker', region_name=region)
sm_runtime = boto3.client('sagemaker-runtime', region_name=region)

print(f'Training prefix: {training_job_prefix}')
print(f'Model prefix:    {model_prefix}')
print(f'Endpoint:        {endpoint_name}')
print(f'Region:          {region}')

## 1. Check Latest Training Job

In [ ]:
resp = sm.list_training_jobs(SortBy='CreationTime', SortOrder='Descending', MaxResults=5)
matched = [j for j in resp['TrainingJobSummaries'] if j['TrainingJobName'].startswith(training_job_prefix)]

assert matched, f'No training jobs found with prefix: {training_job_prefix}'
latest = matched[0]
job_name = latest['TrainingJobName']
print(f'Latest training job: {job_name}')
print(f'Status:             {latest["TrainingJobStatus"]}')
print(f'Created:            {latest["CreationTime"]}')

# Get metrics
details = sm.describe_training_job(TrainingJobName=job_name)
metrics = details.get('FinalMetricDataList', [])
if metrics:
    print(f'\nMetrics:')
    for m in metrics:
        print(f'  {m["MetricName"]:30s} {m["Value"]:.4f}')

## 2. Check Model Exists

In [ ]:
resp = sm.list_models(SortBy='CreationTime', SortOrder='Descending', MaxResults=5)
matched_models = [m for m in resp['Models'] if m['ModelName'].startswith(model_prefix)]

assert matched_models, f'No models found with prefix: {model_prefix}'
model_name = matched_models[0]['ModelName']
print(f'Latest model: {model_name}')
print(f'Created:      {matched_models[0]["CreationTime"]}')

## 3. Check Endpoint

In [ ]:
try:
    ep = sm.describe_endpoint(EndpointName=endpoint_name)
    print(f'Endpoint:  {endpoint_name}')
    print(f'Status:    {ep["EndpointStatus"]}')
    print(f'Created:   {ep["CreationTime"]}')
    endpoint_ready = ep['EndpointStatus'] == 'InService'
except sm.exceptions.ClientError:
    print(f'Endpoint {endpoint_name} not found')
    endpoint_ready = False

## 4. Test Inference

In [ ]:
if endpoint_ready:
    # Sample input: 20 features (matching the training data)
    sample = '0.5,0.3,0.1,0.2,0.4,0.6,0.8,0.1,0.3,0.5,0.2,0.7,0.4,0.1,0.9,0.3,0.6,0.2,0.5,0.8'
    response = sm_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='text/csv',
        Body=sample
    )
    prediction = response['Body'].read().decode('utf-8').strip()
    print(f'Sample input:  {sample[:50]}...')
    print(f'Prediction:    {prediction}')
    print(f'Inference:     PASSED')
else:
    print('Skipping inference — endpoint not InService')

## 5. Summary

In [ ]:
print('=' * 60)
print('MLOPS VALIDATION SUMMARY')
print('=' * 60)
print(f'  Training Job:  {job_name}')
print(f'  Model:         {model_name}')
print(f'  Endpoint:      {endpoint_name}')
print(f'  Endpoint Ready:{" YES" if endpoint_ready else " NO"}')
print(f'  Status:        PASSED')
print('=' * 60)